In [ ]:
import requests
import json
import csv
from datetime import datetime

# --- Config ---
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "tinyllama"
QUESTIONS_FILE = "questions.json"

SYSTEM_PROMPT = """
You are a financial decision maker evaluating investment choices.
For multiple choice questions, respond in this exact format:
ANSWER: [A or B]
RATIONALE: [exactly 3 sentences explaining your reasoning]

For open-ended questions, respond in this exact format:
RATIONALE: [exactly 3 sentences explaining your overall reasoning]

Stay consistent with your decision-making style throughout.
Do not add anything outside of the format above.
"""

# --- Load questions ---
with open(QUESTIONS_FILE) as f:
    data = json.load(f)


def ask(prompt: str) -> str:
    """Each question is fully independent — no history passed."""
    response = requests.post(OLLAMA_URL, json={
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt}
        ],
        "stream": False
    })
    return response.json()["message"]["content"].strip()


def parse_response(raw: str, is_open_ended: bool = False):
    """Extract answer and rationale from model output."""
    lines = raw.strip().split("\n")
    answer = ""
    rationale = ""

    for line in lines:
        if line.startswith("ANSWER:"):
            answer = line.replace("ANSWER:", "").strip()
        if line.startswith("RATIONALE:"):
            rationale = line.replace("RATIONALE:", "").strip()

    # Grab just the letter in case model adds extra text e.g. "A — because..."
    if answer:
        answer = answer[0].upper()

    return answer, rationale


def format_question(q: dict, instruction: str) -> str:
    """Build the prompt string for a question."""
    prompt = q["prompt"]

    if q["choices"]:
        choices_text = "\n".join([f"{k}) {v}" for k, v in q["choices"].items()])
        return f"{prompt}\n\n{choices_text}\n\n{instruction}"
    else:
        # Open-ended question (e.g. H2_Q11)
        return f"{prompt}\n\n{instruction}"


# --- Run pipeline ---
results = []

for section in data["sections"]:
    section_id = section["section"]
    instruction = section["instruction"]

    print(f"\n{'='*50}")
    print(f"Section: {section_id}")
    print(f"{'='*50}")

    for q in section["questions"]:
        is_open_ended = q["choices"] is None
        prompt = format_question(q, instruction)

        print(f"\n[{q['id']}] {q['prompt']}")

        raw = ask(prompt)
        answer, rationale = parse_response(raw, is_open_ended)

        print(f"  ANSWER: {answer}" if answer else "  (open-ended)")
        print(f"  RATIONALE: {rationale[:80]}...")

        results.append({
            "section":      section_id,
            "question_id":  q["id"],
            "pair":         q.get("pair", ""),
            "frame":        q.get("frame", ""),
            "question":     q["prompt"],
            "answer":       answer,
            "rationale":    rationale,
            "raw_response": raw
        })

# --- Export to CSV ---
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = f"results_{timestamp}.csv"

with open(output_file, "w", newline="", encoding="utf-8") as f:
    fieldnames = ["section", "question_id", "pair", "frame", "question", "answer", "rationale", "raw_response"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

print(f"\n✓ Done. Results saved to {output_file}")
print(f"  Total questions answered: {len(results)}")
